**Project:** Data Mining II (2025/26)

**Group Number:** 12

**Members:**
- Beatriz Boura - 20250272
- Dinis Gaspar - 20221869
- Margarida Cruz - 20221929

**Project Overview**

In recent years, nonprofit organizations have faced a growing challenge: while charitable causes have multiplied, public tolerance for repeated, generic solicitations has significantly decreased, often leading to donor fatigue and long-term disengagement. To address this issue, the Civic Support Alliance (CSA)—a federation representing multiple humanitarian and social aid programs—seeks to modernize its fundraising strategy. Rather than launching blanket campaigns across their entire database, the organization aims to transition to a highly targeted approach. The goal is to maximize operational efficiency and maintain donor respect by contacting fewer, but more receptive, individuals. 

As data scientists, our team has been tasked with building a predictive machine learning system using historical demographic, interaction, and donation data accumulated from past campaigns. The primary objective is to accurately answer a fundamental question: Will this person donate if contacted? 

**Notebook Introduction**

In this notebook, we will develop K-Nearest Neighbors models, we will find parameter regions to test and we will then perform a parameter search to try and maximize performances.

**Benchmarks**

As the goal of this project is to help the CSA target the appro

# Imports

In [ ]:
%cd ..
import numpy as np
import pandas as pd
from sklearn.model_selection import TunedThresholdClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.compose import make_column_selector
from sklearn.neighbors import KNeighborsClassifier
import pickle


c:\Users\dinis\OneDrive\Ambiente de Trabalho\Faculdade - MGI-BI\1º ano\2º Semestre\Data Mining II\Project\DM2_Project


In [ ]:
train = pd.read_csv('Files/donors_train.csv')
with open('Files\Pickle Files\model_testing_skf.pkl', 'rb') as file:
    model_testing_skf = pickle.load(file)
with open('Files\Pickle Files\X_train_preprocessed.pkl', 'rb') as file:
    X_train_preprocessed = pickle.load(file)
with open('Files\Pickle Files\X_val_preprocessed.pkl', 'rb') as file:
    X_val_preprocessed = pickle.load(file)

In [ ]:
X = train.drop('TARGET_B', axis=1)
y = train['TARGET_B']

# Defining the Pipeline

We're first going to start by defining the pipeline we're gonna use as a base. This is the pipeline introduced in the Modeling Tools notebook.

In [ ]:
# Categorical Feature Sub-Pipeline
# This is the part that handles the categorical columns, performing
# mode imputation, feature selection using our custom CategoricalFeatureSelector
# and finally one-hot encoding the features.
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('feature_selection',  CategoricalFeatureSelector()),
    # 3. Your specialized encoding (OneHot/Target) now receives imputed integers
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first')),
])

# Numerical Feature Sub-Pipeline
# Here we take care of our numerical features, starting with outlier clipping and feature
# creation using our custom transformer, then scaling data, so that it can then 
# be imputed and numerical feature selection can be performed
num_pipe = Pipeline([
    ('clipper', OutlierClipper()),
    ('feature_engineer', FeatureEngineer()),
    ('scaler', RobustScaler()),
    ('imputer', KNNImputer()),
    ('feature_selection', NumericalFeatureSelector(random_state=SEED))
])

# Here we use a ColumnTransformer with column selectors to perform the split
# between numerical and categorical data, so that each subset can be directed
# to the appropriate sub-pipeline
preprocessor = ColumnTransformer([
    ('cat_section', cat_pipe, make_column_selector(dtype_exclude=[np.number])),
    ('num_section', num_pipe, make_column_selector(dtype_include=[np.number])),
],
verbose_feature_names_out=False)
#  

 
# Final Pipeline
knn_pipe = Pipeline([
    ('cleaner', data_cleaner),
    ('preprocessing', preprocessor),
    ('model', TunedThresholdClassifierCV(DecisionTreeClassifier(),
                                         n_jobs=-1,
                                         scoring='f1',
                                         random_state=SEED
                                         )) 
])

# We use the set_output to pandas so that intermediate transformers can use column
# names, since some of the default scikit-learn transformers by default return 
# Numpy arrays, which obviously don't have column names.
knn_pipe.set_output(transform="pandas")

# Number of Neighbors
The first parameter we're going to look at is the number of neighbors, given the relatively large size of the dataset and the level of complexity of the problem we're going test 1 to 150 to get the best picture we can.

In [ ]:
k_list = np.arange(1, 150)
scores_train = []
scores_val = []
high_score=0
nof=0
for k in tqdm(k_list):
    knn_model = TunedThresholdClassifierCV(KNeighborsClassifier(n_neighbors=k,
                                                                n_jobs=-1),
                                           n_jobs=-1,
                                           scoring='f1')
    knn_model.fit(X_train_t, y_train)
    train_pred = knn_model.predict(X_train_t)
    val_pred = knn_model.predict(X_val_t)
    scores_train.append(f1_score(y_train, train_pred))
    scores_val.append(f1_score(y_val, val_pred))
    if(f1_score(y_val, val_pred)>high_score):
        high_score = f1_score(y_val, val_pred)
        nof = k_list[k-1]


print("Best number of neighbors: %d" %nof)
print("Mean F1 score in train with %d neighbors: %f" % (nof, scores_train[nof-1]))
print("Mean F1 score in validation with %d neighbors: %f" % (nof, high_score))
plt.plot(k_list, scores_train, label='Train')
plt.plot(k_list, scores_val, label = 'Validation')
plt.vlines(x=nof,ymax=high_score,ymin=min(scores_val),ls='--',colors='g')
plt.xticks(k_list)
plt.xlabel('k')
plt.ylabel('score')
plt.legend()

plt.show()